In [ ]:
!pip install transformers datasets seqeval accelerate
!pip install py_vncorenlp
!pip install evaluate seqeval

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [3]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModel
import os
import json
from datasets import Dataset
import py_vncorenlp
from sklearn.model_selection import train_test_split

# Load data

In [4]:
path = '/content/drive/MyDrive/NLP/Data/Offical Data'
data_path1 = os.path.join(path, 'data_label_1.csv')
data_path2 = os.path.join(path, 'data_label_2.csv')
data_path3 = os.path.join(path, 'data_label_3.csv')

In [5]:
df1 = pd.read_csv(data_path1)
df1.drop(columns=['annotation_id', 'annotator', 'created_at','id','updated_at','lead_time'], inplace=True)
df1.head()

,label,text
0,"[{""start"":22,""end"":34,""text"":""88 play 2024"",""l...",cần pass vợt cầu_lông 88 play 2024 đánh được 2...
1,"[{""start"":0,""end"":12,""text"":""Vợt cầu_lông"",""la...","Vợt cầu_lông Haojian màu đen , cam , đỏ , thiế..."
2,"[{""start"":0,""end"":12,""text"":""Vợt cầu_lông"",""la...",Vợt cầu_lông Prokennex_Stormbreaker 618 màu đe...
3,"[{""start"":3,""end"":6,""text"":""vợt"",""labels"":[""PR...",Xả vợt Sale mạnh cho ae chơi tết Đủ các mã vợt...
4,"[{""start"":14,""end"":17,""text"":""vợt"",""labels"":[""...",Mình cần pass vợt Victor_TK 220h II 99% cước 1...


In [6]:
df2 = pd.read_csv(data_path2)
df2.drop(columns=['annotation_id', 'annotator', 'created_at','id','updated_at','lead_time'], inplace=True)
df2.head()

,label,text
0,"[{""start"":314,""end"":322,""text"":""21000000"",""lab...",Laptop_Acer_Nitro_V ANV 15-52 - Bảo_hành chính...
1,"[{""start"":447,""end"":534,""text"":""D \/ C : 28 đư...",Laptop_Xiaomi_Redmibook_Pro 15 2022 màu xám : ...
2,"[{""start"":8,""end"":24,""text"":""Surface laptop 3""...",Em pass Surface laptop 3 i 5-103 5G 7 RAM 8Gb ...
3,"[{""start"":0,""end"":13,""text"":""SURFACE PRO 8"",""l...",SURFACE PRO 8 ( i 5/8 / 256 ) Giá chỉ từ : 100...
4,"[{""start"":303,""end"":311,""text"":""10000000"",""lab...",Laptop_Asus_Vivobook_Pro 15 OLED M6500QC Gamin...


In [7]:
df3 = pd.read_csv(data_path3)
df3.drop(columns=['annotation_id', 'annotator', 'created_at','id','updated_at','lead_time'], inplace=True)
df3.head()

,label,text
0,"[{""start"":22,""end"":33,""text"":""Samsung S21"",""la...","giá 5900000 mình pass Samsung S21 5G 8/128 ạ ,..."
1,"[{""start"":64,""end"":74,""text"":""điện_thoại"",""lab...",gdtt Hải_Dương giá 350000 vnd mình thanh_lý má...
2,"[{""start"":9,""end"":35,""text"":""phông_nền chụp ản...","mình bán phông_nền chụp ảnh 1.6 x2m ạ , màu tr..."
3,"[{""start"":31,""end"":58,""text"":""máy đóng gáy lò_...",gdtt Cầu_Giấy Hà_Nội mình pass máy đóng gáy lò...
4,"[{""start"":25,""end"":41,""text"":"" iPhone 13 128gb...","giá 8500000 mình thanh_lý iPhone 13 128gb ạ , ..."


In [8]:
df = pd.concat([df1, df2, df3], ignore_index=True)
df.head()

,label,text
0,"[{""start"":22,""end"":34,""text"":""88 play 2024"",""l...",cần pass vợt cầu_lông 88 play 2024 đánh được 2...
1,"[{""start"":0,""end"":12,""text"":""Vợt cầu_lông"",""la...","Vợt cầu_lông Haojian màu đen , cam , đỏ , thiế..."
2,"[{""start"":0,""end"":12,""text"":""Vợt cầu_lông"",""la...",Vợt cầu_lông Prokennex_Stormbreaker 618 màu đe...
3,"[{""start"":3,""end"":6,""text"":""vợt"",""labels"":[""PR...",Xả vợt Sale mạnh cho ae chơi tết Đủ các mã vợt...
4,"[{""start"":14,""end"":17,""text"":""vợt"",""labels"":[""...",Mình cần pass vợt Victor_TK 220h II 99% cước 1...


# Tagging

In [9]:
def tagging(full_text, annotation):

  words = []
  word_index = []
  current_index = 0
  for word in full_text.split(" "):
    word_start = full_text.find(word, current_index)
    word_end = word_start + len(word)
    words.append(word)
    word_index.append((word_start, word_end))
    current_index = word_end
    tag_seq = ['O']*len(words)
  anno_list = json.loads(annotation)
  for ann in anno_list:
    ann_start = ann['start']
    ann_end = ann['end']
    ann_type = ann['labels'][0]

    entity_start = False
    for i, (w_start, w_end) in enumerate(word_index):
      if ann_start <= w_start and w_end <= ann_end:
        if not entity_start:
          tag_seq[i] = f"B-{ann_type}"
          entity_start = True
        else:
          tag_seq[i] = f"I-{ann_type}"
  return  tag_seq

In [10]:
df['tag_seg'] = df.apply(lambda x: tagging(x['text'], x['label']), axis=1)

In [11]:
df

,label,text,tag_seg
0,"[{""start"":22,""end"":34,""text"":""88 play 2024"",""l...",cần pass vợt cầu_lông 88 play 2024 đánh được 2...,"[O, O, B-PROUCT_TYPE, I-PROUCT_TYPE, B-PROUCT_..."
1,"[{""start"":0,""end"":12,""text"":""Vợt cầu_lông"",""la...","Vợt cầu_lông Haojian màu đen , cam , đỏ , thiế...","[B-PROUCT_TYPE, I-PROUCT_TYPE, B-PROUCT_NAME, ..."
2,"[{""start"":0,""end"":12,""text"":""Vợt cầu_lông"",""la...",Vợt cầu_lông Prokennex_Stormbreaker 618 màu đe...,"[B-PROUCT_TYPE, I-PROUCT_TYPE, B-PROUCT_NAME, ..."
3,"[{""start"":3,""end"":6,""text"":""vợt"",""labels"":[""PR...",Xả vợt Sale mạnh cho ae chơi tết Đủ các mã vợt...,"[O, B-PROUCT_TYPE, O, O, O, O, O, O, O, O, O, ..."
4,"[{""start"":14,""end"":17,""text"":""vợt"",""labels"":[""...",Mình cần pass vợt Victor_TK 220h II 99% cước 1...,"[O, O, O, B-PROUCT_TYPE, B-PROUCT_NAME, I-PROU..."
...,...,...,...
1217,"[{""start"":27,""end"":55,""text"":""bộ kim chỉ may_v...",giá 80000 Bắc_Ninh em pass bộ kim chỉ may_vá +...,"[O, B-PRICE, B-LOCATION, O, O, B-PROUCT_NAME, ..."
1218,"[{""start"":14,""end"":17,""text"":""kệ "",""labels"":[""...","chị mình pass kệ đựng mỹ_phẩm xoay 360 ạ , để ...","[O, O, O, B-PROUCT_NAME, I-PROUCT_NAME, I-PROU..."
1219,"[{""start"":29,""end"":37,""text"":""hộp đựng"",""label...",giá 60000 Quận 8 HCM em pass hộp đựng thuốc mi...,"[O, B-PRICE, B-LOCATION, I-LOCATION, I-LOCATIO..."
1220,"[{""start"":30,""end"":47,""text"":""máy khử mùi phòn...",Thanh_Hoá giá 220000 mình bán máy khử mùi phòn...,"[B-LOCATION, O, B-PRICE, O, O, B-PROUCT_NAME, ..."


# Train Test Split

In [12]:
trainning_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(trainning_df, test_size=0.2, random_state=42)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [13]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(781, 3)
(196, 3)
(245, 3)


In [14]:
train_data = Dataset.from_pandas(train_df)
val_data = Dataset.from_pandas(val_df)
test_data = Dataset.from_pandas(test_df)

# Tag Vocab

In [15]:

class Vocabulary:
    def __init__(self, token_to_idx=None, use_unk=True):
        """
        Args:
            token_to_idx (dict): a pre-existing map of tokens to indices
        """
        if token_to_idx is None:
            token_to_idx = {}
        self._token_to_idx = token_to_idx

        self._idx_to_token = {idx: token
                              for token, idx in self._token_to_idx.items()}

        self.pad_index = 0

        if use_unk:
            self.unk_index = 1
        else:
            self.unk_index = -1

    def lookup_token(self, token):
        """Retrieve the index associated with the token
          or the UNK index if token isn't present.

        Args:
            token (str): the token to look up
        Returns:
            index (int): the index corresponding to the token
        Notes:
            `unk_index` needs to be >=0 (having been added into the Vocabulary)
              for the UNK functionality
        """
        if self.unk_index >= 0:
            return self._token_to_idx.get(token, self.unk_index)
        else:
            return self._token_to_idx[token]

    def lookup_index(self, index):
        """Return the token associated with the index

        Args:
            index (int): the index to look up
        Returns:
            token (str): the token corresponding to the index
        Raises:
            KeyError: if the index is not in the Vocabulary
        """
        if index not in self._idx_to_token:
            raise KeyError("the index (%d) is not in the Vocabulary" % index)
        return self._idx_to_token[index]

    def add_token(self, token):
        """Update mapping dicts based on the token.

        Args:
            token (str): the item to add into the Vocabulary
        Returns:
            index (int): the integer corresponding to the token
        """
        if token in self._token_to_idx:
            index = self._token_to_idx[token]
        else:
            index = len(self._token_to_idx)
            self._token_to_idx[token] = index
            self._idx_to_token[index] = token
        return index

    @classmethod
    def build_vocab(cls, sequences, use_unk=True):
        """Build vocabulary from a list of sequences
        A sequence may be a sequence of words or a sequence of tags.

        Arguments:
        ----------
            sequences (list): list of sequences, each sentence list of words
            or list of tags

        Return:
        ----------
            vocab (Vocabulary): a Vocabulary object
        """
        if use_unk:
            token_to_idx = {"<PAD>": 0, "<UNK>": 1}
        else:
            token_to_idx = {"<PAD>": 0}

        vocab = cls(token_to_idx, use_unk=use_unk)
        for s in sequences:
            for word in s:
                vocab.add_token(word)
        return vocab

    def __str__(self):
        return "<Vocabulary(size=%d)>" % len(self)

    def __len__(self):
        return len(self._token_to_idx)

In [16]:
tag_vocab = Vocabulary.build_vocab(df['tag_seg'])

In [17]:
print(tag_vocab._token_to_idx)

{'<PAD>': 0, '<UNK>': 1, 'O': 2, 'B-PROUCT_TYPE': 3, 'I-PROUCT_TYPE': 4, 'B-PROUCT_NAME': 5, 'I-PROUCT_NAME': 6, 'B-PRICE': 7, 'B-LOCATION': 8, 'I-LOCATION': 9, 'I-PRICE': 10}


# Tokenizer

In [ ]:

tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

In [25]:
def tokenize_and_align_labels(df):
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []

    for i in range(len(df["text"])):

        words = df["text"][i].split()

        ner_tags = df["tag_seg"][i]
        if isinstance(ner_tags, str):
            import ast
            ner_tags = ast.literal_eval(ner_tags)

        input_ids = [tokenizer.cls_token_id]
        label_ids = [-100] # Token <s>

        for word, label in zip(words, ner_tags):
            sub_words = tokenizer.tokenize(word)
            if not sub_words: # Bỏ qua nếu từ trống
                continue

            sub_word_ids = tokenizer.convert_tokens_to_ids(sub_words)
            input_ids.extend(sub_word_ids)

            label_ids.append(tag_vocab._token_to_idx[label])
            label_ids.extend([-100] * (len(sub_words) - 1))


        input_ids.append(tokenizer.sep_token_id)
        label_ids.append(-100)

        attention_mask = [1] * len(input_ids)

        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(label_ids)

    return {
        "input_ids": batch_input_ids,
        "attention_mask": batch_attention_mask,
        "labels": batch_labels
    }

In [ ]:
tokenized_train = train_data.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_data.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_data.map(tokenize_and_align_labels, batched=True)

In [27]:
cols_to_remove = train_data.column_names

tokenized_train = tokenized_train.remove_columns(cols_to_remove)
tokenized_val = tokenized_val.remove_columns(cols_to_remove)
tokenized_test = tokenized_test.remove_columns(cols_to_remove)

print(tokenized_train.column_names)

['input_ids', 'attention_mask', 'labels']


# Training

In [28]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [ ]:
pho_bert = AutoModelForTokenClassification.from_pretrained(
    "vinai/phobert-base-v2",
    num_labels=len(tag_vocab._token_to_idx),
    id2label = tag_vocab._idx_to_token,
    label2id = tag_vocab._token_to_idx)


In [ ]:
import evaluate

seqeval = evaluate.load("seqeval")
label_list = list(tag_vocab._token_to_idx.keys())

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]


    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./phobert_ner_model",
    eval_strategy="epoch",      # Evaluate on Validation for each epoch
    save_strategy="epoch",            # Save weight after each epoch
    learning_rate=2e-5,               # Standard learning reate
    per_device_train_batch_size=16,   # Batch size
    per_device_eval_batch_size=16,
    num_train_epochs=15,
    weight_decay=0.01,
    load_best_model_at_end=True,      # Save the best
    metric_for_best_model="f1",       # Best metric f1
)


trainer = Trainer(
    model=pho_bert,
    args=training_args,
    train_dataset=tokenized_train,    # Train
    eval_dataset=tokenized_val,       # Validation
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

In [32]:
# Evaluate on test set
test_results = trainer.predict(tokenized_test)
metrics = test_results.metrics

print(f"Precision: {metrics['test_precision']:.4f}")
print(f"Recall: {metrics['test_recall']:.4f}")
print(f"F1-Score: {metrics['test_f1']:.4f}")

Precision: 0.7809
Recall: 0.8251
F1-Score: 0.8024
